# 02 -- Marcgravia evenia reconstruction

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/antnewman/acoustic-beacon-optimiser/blob/main/notebooks/02_marcgravia_reconstruction.ipynb)

Compute the acoustic directivity of a spherical-cap approximation to the
*Marcgravia evenia* dish-leaf (diameter approximately 35 mm, depth approximately 25 mm,
Simon et al., 2011) and compare its spectral directional pattern against the
published empirical pattern.

The geometry assumed here is a spherical cap; Simon et al. (2011) characterise
the dish-leaf as approximately spherical over its central region. Future
iterations of this notebook should substitute a scanned or photogrammetrically
reconstructed profile.

**Runtime:** a few minutes per frequency on a single CPU; the full spectral
sweep below takes of order 30 minutes.

**Setup in Colab:** run `!pip install -q git+https://github.com/antnewman/acoustic-beacon-optimiser.git` in the first cell, then continue.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from abo.acoustics.target_strength import monostatic_target_strength
from abo.geometry.meshing import mesh_spherical_cap
from abo.geometry.reflectors import SphericalCap
from abo.visualisation.spectral_plots import (
    plot_on_axis_vs_frequency,
    plot_ts_heatmap,
    plot_ts_polar,
)

## Build the Marcgravia-approximating spherical cap

Published dimensions (Simon et al., 2011): diameter approximately 35 mm, depth approximately 25 mm.
The aperture radius is therefore 17.5 mm and depth 25 mm. This uniquely determines
a spherical cap: the sphere radius follows from `r = (d/2)^2/(2*depth) + depth/2`.

In [ ]:
aperture_m = 0.0175
depth_m = 0.025
# From d^2/4 + (r - h)^2 = r^2  =>  r = (aperture^2 + depth^2) / (2*depth)
sphere_r = (aperture_m**2 + depth_m**2) / (2.0 * depth_m)
half_angle = float(np.arcsin(aperture_m / sphere_r))
cap = SphericalCap(radius=sphere_r, half_angle=half_angle)
print(f"Sphere radius: {sphere_r*1000:.2f} mm")
print(f"Half angle:    {np.degrees(half_angle):.2f} deg")
print(f"Depth (check): {cap.depth*1000:.2f} mm")
print(f"Aperture:      {cap.aperture*1000:.2f} mm")
print(f"Surface area:  {cap.surface_area*1e6:.1f} mm^2")

## Spectral sweep

In [ ]:
frequencies = np.linspace(30_000.0, 120_000.0, 16)
angles = np.linspace(0.0, np.pi / 2, 13)

grid = mesh_spherical_cap(cap, frequency=float(frequencies.max()))
print(f"Mesh: {grid.number_of_elements} elements")

ts = monostatic_target_strength(
    grid, frequencies, angles, far_field_range=1.0,
)
print(f"TS range: [{ts.min():.1f}, {ts.max():.1f}] dB")

## Figures

In [ ]:
fig = plot_ts_heatmap(
    ts, frequencies, angles,
    title="Marcgravia evenia approximation: monostatic TS",
)
fig.savefig("../paper/figures/fig02_marcgravia_ts_heatmap.png", dpi=200)
plt.show()

In [ ]:
fig = plot_on_axis_vs_frequency(
    ts, frequencies, angles,
    title="Marcgravia evenia approximation: on-axis TS",
)
fig.savefig("../paper/figures/fig02_marcgravia_onaxis.png", dpi=200)
plt.show()

In [ ]:
# Polar directivity at a representative glossophagine frequency
i_ref = int(np.argmin(np.abs(frequencies - 65_000.0)))
fig = plot_ts_polar(ts[i_ref, :], angles, frequencies[i_ref])
fig.savefig("../paper/figures/fig02_marcgravia_polar.png", dpi=200)
plt.show()